# Train four ModernBERT classifiers and export them

Run from the repository root after notebooks 4, 5 and 8 have produced audited labels and matching manifests. Notebook 7 is descriptive analysis; there is no notebook 6, and the classical-baseline notebook 9 is intentionally excluded from this transformer-only release.

The model head is built from the retained classes in training_manifest.json. The old root-cause checkpoint has 26 outputs and no verified matching label order. Do not deploy it or use ignore_mismatched_sizes to pretend it was repaired. Fresh training from ModernBERT-base is the supported correction.

Run setup cells, then each task's train and evaluate cells. Run one task at a time. Exports go to models/transformer_exports/<task>/<timestamp>. Use README.md to select the four exports and start Streamlit. Risk remains five-band classification with a disclosed numerical conversion at inference.


In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from datasets import DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# Shared by every task; run setup before choosing a training section.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

import gc


In [ ]:
# ── Configuration ──
MODEL_NAME = "answerdotai/ModernBERT-base"
TOKENIZED_DIR = "data/tokenized"
MODEL_DIR = "models"
Path(MODEL_DIR).mkdir(exist_ok=True)

TASKS = {
    "sentiment": {
        "num_labels": None,
        "encoder_path": "data/model_ready/encoders/le_sentiment.pkl",
        "epochs": 5,
        "batch_size": 8,
        "lr": 2e-5,
    },
    "intent": {
        "num_labels": None,  
        "encoder_path": "data/model_ready/encoders/le_intent.pkl",
        "epochs": 8,
        "batch_size": 8,
        "lr": 2e-5,
    },
    "root_cause": {
        "num_labels": None,
        "encoder_path": "data/model_ready/encoders/le_root_cause.pkl",
        "epochs": 8,
        "batch_size": 8,
        "lr": 2e-5,
    },
    "risk": {
        "num_labels": 5,
        "encoder_path": None,  
        "epochs": 5,
        "batch_size": 8,
        "lr": 2e-5,
    },
}

# The tokenization manifest is authoritative; original encoders may include excluded classes.
for task_name, task_cfg in TASKS.items():
    manifest_path = Path(TOKENIZED_DIR) / task_name / "training_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Run updated notebook 8 first: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 2:
        raise ValueError("Old tokenized data; rerun updated notebook 8")
    task_cfg["label_names"] = manifest["label_names"]
    task_cfg["num_labels"] = len(manifest["label_names"])
    print(task_name, task_cfg["num_labels"], "supported classes")

# New exports are versioned; original *_best folders are not overwritten.
EXPORT_DIR = Path(MODEL_DIR) / "transformer_exports"
# Leave None for fresh training. Set an exact matching checkpoint only to resume
# a run with the SAME labels, data, model and training settings.
RESUME_CHECKPOINTS = {task_name: None for task_name in TASKS}

# Memory-aware defaults: effective batch 8*4=32 on one device.
GRADIENT_ACCUMULATION_STEPS = 4
EVAL_BATCH_SIZE = 8
USE_BF16 = device == "cuda" and torch.cuda.is_bf16_supported()
USE_FP16 = device == "cuda" and not USE_BF16


In [ ]:
# Load saved tokenizers/settings from notebook 8, not guessed deployment defaults.
from datetime import datetime, timezone
import hashlib
import importlib.metadata

def validate_training_manifest(task_name, task_cfg, manifest):
    if manifest.get("schema_version") != 2 or manifest.get("task") != task_name:
        raise ValueError(f"{task_name}: invalid training manifest; rerun notebook 8.")
    if manifest.get("base_model") != MODEL_NAME:
        raise ValueError(f"{task_name}: tokenizer model changed; rerun notebook 8.")
    if manifest.get("label_names") != task_cfg["label_names"]:
        raise ValueError(f"{task_name}: label order changed; rerun notebook 8.")
    length = manifest.get("max_length")
    if type(length) is not int or length <= 0:
        raise ValueError(f"{task_name}: invalid tokenization length.")
    if manifest.get("text_preprocessing") != "as_is":
        raise ValueError(f"{task_name}: unsupported text preprocessing.")
    return manifest

def load_task_data(task_name):
    folder = Path(TOKENIZED_DIR) / task_name
    manifest_path = folder / "training_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"{manifest_path} missing. Run updated notebook 8 first; do not guess old label mappings.")
    manifest = validate_training_manifest(task_name, TASKS[task_name], json.loads(manifest_path.read_text(encoding="utf-8")))
    source = Path("data/model_ready") / f"{task_name}_dataset.csv"
    if hashlib.sha256(source.read_bytes()).hexdigest() != manifest.get("source_sha256"):
        raise ValueError("Source dataset changed since tokenization; rerun notebook 8.")
    dataset = DatasetDict.load_from_disk(str(folder))
    for split, values in dataset.items():
        labels = np.asarray(values["labels"])
        if not len(labels) or not np.isin(labels, range(TASKS[task_name]["num_labels"])).all():
            raise ValueError(f"{task_name}/{split}: empty split or invalid class IDs.")
        if set(labels.tolist()) != set(range(TASKS[task_name]["num_labels"])):
            raise ValueError(f"{task_name}/{split}: class has zero examples; rerun notebook 8.")
        if len(values) != manifest["split_sizes"][split]:
            raise ValueError(f"{task_name}/{split}: dataset size differs from manifest.")
    task_tokenizer = AutoTokenizer.from_pretrained(str(folder / "tokenizer"), local_files_only=True)
    return dataset, task_tokenizer, manifest

def export_for_website(trainer, task_tokenizer, task_name, manifest):
    validate_training_manifest(task_name, TASKS[task_name], manifest)
    names = TASKS[task_name]["label_names"]
    actual = [trainer.model.config.id2label[i] for i in range(trainer.model.config.num_labels)]
    if actual != names:
        raise ValueError("Model label mapping does not match training data; export stopped.")
    version = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    destination = EXPORT_DIR / task_name / version
    destination.mkdir(parents=True, exist_ok=False)
    # Trainer holds the best validation model after load_best_model_at_end.
    trainer.save_model(str(destination))
    task_tokenizer.save_pretrained(str(destination))
    weights = sorted(destination.glob("*.safetensors")) + sorted(destination.glob("pytorch_model*.bin"))
    if not weights:
        raise FileNotFoundError("Export has no weights; do not deploy this folder.")
    hashes = {}
    for file in weights:
        digest = hashlib.sha256()
        with file.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        hashes[file.name] = digest.hexdigest()
    metadata = dict(manifest)
    metadata.update({
        "model_version": f"modernbert-{task_name}-{version}",
        "output_type": "risk_band" if task_name == "risk" else "class_label",
        "score_conversion": None,
        "weight_sha256": hashes,
        "best_validation_metric": trainer.state.best_metric,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "packages": {name: importlib.metadata.version(name) for name in ("torch", "transformers", "datasets", "numpy")},
    })
    # Written last: absence means the export did not finish.
    (destination / "deployment.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Website export saved: {destination}")
    return destination

def validate_resume(task_name, checkpoint, manifest):
    if checkpoint is None: return
    config = json.loads((Path(checkpoint) / "config.json").read_text(encoding="utf-8"))
    mapping = config.get("id2label", {})
    if [mapping.get(str(i)) for i in range(len(mapping))] != TASKS[task_name]["label_names"]:
        raise ValueError("Resume checkpoint labels differ; use a matching checkpoint or fresh training.")

    expected_hash = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    if config.get("training_manifest_sha256") != expected_hash:
        raise ValueError("Resume checkpoint belongs to different/unverified tokenized data; start fresh.")


## Task 1: Sentiment Classification
Starting with the simplest task (4 classes) to validate the training pipeline.

In [ ]:
# ── Load tokenized sentiment data ──
task = "sentiment"
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

In [ ]:
# ── Train sentiment model ──
task = "sentiment"
task_cfg = TASKS[task]
# Reload task data here too, so running a training cell out of order cannot use another task.
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Release the previous task before allocating the next transformer.
if "trainer" in globals(): del trainer
if "model" in globals(): del model
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
    id2label=dict(enumerate(task_cfg["label_names"])),
    label2id={label: i for i, label in enumerate(task_cfg["label_names"])},
)

model.config.training_manifest_sha256 = hashlib.sha256(json.dumps(training_manifest, sort_keys=True).encode()).hexdigest()

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    seed=42,
    data_seed=42,
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    eval_accumulation_steps=8,
    warmup_ratio=0.06,
    save_safetensors=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training {task}...")
validate_resume(task, RESUME_CHECKPOINTS[task], training_manifest)
trainer.train(resume_from_checkpoint=RESUME_CHECKPOINTS[task])
print("Training complete!")

# Export the actual best model, exact tokenizer and deployment settings together.
trainer._finsentiment_task = task
export_path = export_for_website(trainer, tokenizer, task, training_manifest)


In [ ]:
# ── Evaluate sentiment on test set ──
task = "sentiment"
task_cfg = TASKS[task]

if getattr(trainer, "_finsentiment_task", None) != task:
    raise ValueError(f"Run the {task} training cell before its evaluation cell.")
preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])


print(f"SENTIMENT — TEST SET RESULTS")

print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"], zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred, labels=list(range(task_cfg["num_labels"])))
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()
# Evaluation belongs to this exact exported version.
evaluation = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    "test_rows": int(len(y_true)),
    "classification_report": classification_report(y_true, y_pred,
        labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"],
        zero_division=0, output_dict=True),
}
(export_path / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")


## Task 2: Intent Classification
Training model for predicting intent classes.

In [ ]:
# ── Load tokenized intent data ──
task = "intent"
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

In [ ]:
# ── Train intent model ──
task = "intent"
task_cfg = TASKS[task]
# Reload task data here too, so running a training cell out of order cannot use another task.
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Release the previous task before allocating the next transformer.
if "trainer" in globals(): del trainer
if "model" in globals(): del model
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
    id2label=dict(enumerate(task_cfg["label_names"])),
    label2id={label: i for i, label in enumerate(task_cfg["label_names"])},
)

model.config.training_manifest_sha256 = hashlib.sha256(json.dumps(training_manifest, sort_keys=True).encode()).hexdigest()

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    seed=42,
    data_seed=42,
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    eval_accumulation_steps=8,
    warmup_ratio=0.06,
    save_safetensors=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training {task}...")
validate_resume(task, RESUME_CHECKPOINTS[task], training_manifest)
trainer.train(resume_from_checkpoint=RESUME_CHECKPOINTS[task])
print("Training complete!")

# Export the actual best model, exact tokenizer and deployment settings together.
trainer._finsentiment_task = task
export_path = export_for_website(trainer, tokenizer, task, training_manifest)


In [ ]:
# ── Evaluate intent on test set ──
task = "intent"
task_cfg = TASKS[task]

if getattr(trainer, "_finsentiment_task", None) != task:
    raise ValueError(f"Run the {task} training cell before its evaluation cell.")
preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])

print("=" * 60)
print(f"INTENT — TEST SET RESULTS")
print("=" * 60)
print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"], zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred, labels=list(range(task_cfg["num_labels"])))
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation='vertical')
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()
# Evaluation belongs to this exact exported version.
evaluation = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    "test_rows": int(len(y_true)),
    "classification_report": classification_report(y_true, y_pred,
        labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"],
        zero_division=0, output_dict=True),
}
(export_path / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")


## Task 3: Root Cause Classification
Training model for predicting root causes.

In [ ]:
# ── Load tokenized root_cause data ──
task = "root_cause"
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

In [ ]:
# ── Train root_cause model ──
task = "root_cause"
task_cfg = TASKS[task]
# Reload task data here too, so running a training cell out of order cannot use another task.
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Release the previous task before allocating the next transformer.
if "trainer" in globals(): del trainer
if "model" in globals(): del model
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
    id2label=dict(enumerate(task_cfg["label_names"])),
    label2id={label: i for i, label in enumerate(task_cfg["label_names"])},
)

model.config.training_manifest_sha256 = hashlib.sha256(json.dumps(training_manifest, sort_keys=True).encode()).hexdigest()

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    seed=42,
    data_seed=42,
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    eval_accumulation_steps=8,
    warmup_ratio=0.06,
    save_safetensors=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training {task}...")
validate_resume(task, RESUME_CHECKPOINTS[task], training_manifest)
trainer.train(resume_from_checkpoint=RESUME_CHECKPOINTS[task])
print("Training complete!")

# Export the actual best model, exact tokenizer and deployment settings together.
trainer._finsentiment_task = task
export_path = export_for_website(trainer, tokenizer, task, training_manifest)


In [ ]:
# ── Evaluate root_cause on test set ──
task = "root_cause"
task_cfg = TASKS[task]

if getattr(trainer, "_finsentiment_task", None) != task:
    raise ValueError(f"Run the {task} training cell before its evaluation cell.")
preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])

print(f"ROOT CAUSE — TEST SET RESULTS")

print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"], zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_true, y_pred, labels=list(range(task_cfg["num_labels"])))
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d", xticks_rotation='vertical')
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()
# Evaluation belongs to this exact exported version.
evaluation = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    "test_rows": int(len(y_true)),
    "classification_report": classification_report(y_true, y_pred,
        labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"],
        zero_division=0, output_dict=True),
}
(export_path / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")


## Task 4: Risk Band Classification
Training model for predicting 5 risk bands.

In [ ]:
# ── Load tokenized risk data ──
task = "risk"
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(f"Loaded {task}:")
print(f"  train     : {len(dd['train']):,}")
print(f"  validation: {len(dd['validation']):,}")
print(f"  test      : {len(dd['test']):,}")

In [ ]:
# ── Train risk model ──
task = "risk"
task_cfg = TASKS[task]
# Reload task data here too, so running a training cell out of order cannot use another task.
dd, tokenizer, training_manifest = load_task_data(task)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Release the previous task before allocating the next transformer.
if "trainer" in globals(): del trainer
if "model" in globals(): del model
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=task_cfg["num_labels"],
    id2label=dict(enumerate(task_cfg["label_names"])),
    label2id={label: i for i, label in enumerate(task_cfg["label_names"])},
)

model.config.training_manifest_sha256 = hashlib.sha256(json.dumps(training_manifest, sort_keys=True).encode()).hexdigest()

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/{task}_checkpoints",
    num_train_epochs=task_cfg["epochs"],
    per_device_train_batch_size=task_cfg["batch_size"],
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=task_cfg["lr"],
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    seed=42,
    data_seed=42,
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    eval_accumulation_steps=8,
    warmup_ratio=0.06,
    save_safetensors=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dd["train"],
    eval_dataset=dd["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"Training {task}...")
validate_resume(task, RESUME_CHECKPOINTS[task], training_manifest)
trainer.train(resume_from_checkpoint=RESUME_CHECKPOINTS[task])
print("Training complete!")

# Export the actual best model, exact tokenizer and deployment settings together.
trainer._finsentiment_task = task
export_path = export_for_website(trainer, tokenizer, task, training_manifest)


In [ ]:
# ── Evaluate risk on test set ──
task = "risk"
task_cfg = TASKS[task]

if getattr(trainer, "_finsentiment_task", None) != task:
    raise ValueError(f"Run the {task} training cell before its evaluation cell.")
preds_output = trainer.predict(dd["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = np.array(dd["test"]["labels"])

print(f"RISK — TEST SET RESULTS")

print(f"Accuracy   : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1   : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_true, y_pred, labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"], zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred, labels=list(range(task_cfg["num_labels"])))
disp = ConfusionMatrixDisplay(cm, display_labels=task_cfg["label_names"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"{task.upper()} — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{MODEL_DIR}/{task}_confusion_matrix.png", dpi=150)
plt.show()
# Evaluation belongs to this exact exported version.
evaluation = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    "test_rows": int(len(y_true)),
    "classification_report": classification_report(y_true, y_pred,
        labels=list(range(task_cfg["num_labels"])), target_names=task_cfg["label_names"],
        zero_division=0, output_dict=True),
}
(export_path / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
